# Break Down point

In [8]:
import numpy as np
from math import log
from tqdm import tqdm
 
from Welsch import AlphaDivergenceAlgo
from Huber import HuberAlgo
from help_functions import generate_linear_model

In [ ]:
N_SAMPLES = 10_000
P_FEATURES = 10
BETA_SCALE = 1_000
OUTLIER_SCALE =10000000
DELTA = 0.01
NUM_REPETITIONS = 5_000
EPSILON_RANGE = np.arange(0.0, 0.3, 0.01)
MAX_ITER = 100
SEED = 100
 
np.random.seed(SEED)
beta_star = BETA_SCALE * np.random.choice([-1, 1], size=P_FEATURES)
tau_base = log(1 / DELTA) / N_SAMPLES

In [ ]:
break_Welsch= []
 
for epsilon in EPSILON_RANGE:
    tau = tau_base + epsilon
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y,X,_ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        model = AlphaDivergenceAlgo(X, y)
        beta_hat = model.optimizer_approach(tau=tau, maxiter=MAX_ITER)
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_alpha_div.append((epsilon, bias, rmse))

In [ ]:
break_down_huber = []
 
for epsilon in EPSILON_RANGE:
    squared_errors = []
    beta_hats = []
    
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        gamma = log(1 / DELTA) / N_SAMPLES + epsilon
        y,X,_ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
        model = HuberAlgo(X, y)
        beta_hat, _ = model.optimizer_approach(gamma=1, max_iter=MAX_ITER)
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_huber.append((epsilon, bias, rmse))
